In [1]:
import numpy as np
import torch
import os
import glob

In [25]:
import pandas as pd

script_file_path = 'bash_scripts/test.sh'
watch_list = []
with open(script_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.startswith('for'):
            continue
        watch = line.split('in')[-1].split()
        watch_list.extend(watch)
print(watch_list)

# Load the config file into a DataFrame
config_file_path = 'bash_scripts/config/test.csv'
df = pd.read_csv(config_file_path)
# Strip leading and trailing spaces from column names
df.columns = df.columns.str.strip()
# Strip leading and trailing spaces from all entries
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
# Iterate over each row in the DataFrame
for idx, row in df.iterrows():
    ckpt_path = f'ckpt/{row["dataset"]}_{row["usage"]}/{row["model"]}_{row["depth"]}/{row["lr"]}_{row["snr"]}_{row["snvr"]}'
    seeds = row["seed"].split(';')

    best_accs = []
    for seed in seeds:
        seed_path = os.path.join(ckpt_path, seed)
        best_epoch_path = os.path.join(seed_path, 'best_epoch.pth')
        if not os.path.exists(best_epoch_path):
            continue
        best_epoch = torch.load(best_epoch_path)
        best_accs.append(best_epoch['test_acc'])        

    if not best_accs:
        continue
    mean_acc = np.mean(best_accs)
    std_acc = np.std(best_accs)
    df.at[idx, 'accuracy'] = f'{mean_acc:.2f} ± {std_acc:.2f} ({len(best_accs)})'

    
df.to_csv(config_file_path, index=False)
df

['5', '6', '7', '8', '9', '4']


/tmp/ipykernel_321339/3130585555.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,dataset,usage,model,depth,lr,snr,snvr,seed,accuracy,remark
0,cifar100,1.0,resnet,18,0.10,0.000,0.0,43;37;13;51;71,NaN,NaN
1,cifar100,1.0,resnet,34,0.05,0.000,0.0,43;37;13;51;71,NaN,NaN
2,cifar100,1.0,vgg,16,0.05,0.000,0.0,43;37;13;51;71,72.21 ± 0.28 (5),NaN
3,cifar100,1.0,vgg_clf,16,0.05,0.000,0.0,43;37;13;51;71,NaN,{'linear_dim':4096}
4,cifar100,1.0,vgg_clf,16,0.05,0.000,0.0,43;37;13;51;71,NaN,{'linear_dim':1024}
5,cifar100,1.0,vgg,16,0.05,0.005,0.0,43;37;13;51;71,NaN,NaN
6,cifar100,1.0,vgg_clf,16,0.05,0.005,0.0,43;37;13;51;71,NaN,{'linear_dim':4096}
7,cifar100,1.0,vgg_clf,16,0.05,0.005,0.0,43;37;13;51;71,NaN,{'linear_dim':1024}
8,tiny-imagenet-200,1.0,resnet,18,0.10,0.000,0.0,43;37;13;51;71,NaN,NaN
